# Hybrid Multi-Agent Simulation: Social Media & Event Dispatch

This notebook demonstrates a practical, lightweight, and non-overengineered hybrid architecture combining **deterministic cognitive orchestration** (User Proxy & Coordinator) with a **reactive pub-sub event layer** (independent consumer agents reacting asynchronously to a feed).

## The Concept: User Proxy Agent vs. Direct Orchestrator
Instead of making a passive user object, we implement a **User Proxy Agent** (or User Agent). 
* **Why is a User Proxy Agent better?** In a pure Actor model runtime, every participant is an Actor with an `AgentId` and a `Mailbox`. A User Proxy represents the human user in the message loop. It can receive messages, aggregate subagent feedback, and forward results back to the user, acting as a clean gateway without hardcoding reasoning logic.

## The Simulation Scenario:
1. **User Proxy Agent** posts a new piece of content (publishes a `new_post` event to a shared feed topic).
2. **Three independent consumer agents** subscribe to the feed topic:
   * **Sentiment Agent**: Analyzes emotional tone.
   * **Moderator Agent**: Performs automated safety/moderation filtering.
   * **SEO Tagging Agent**: Extracts key tags and keywords.
3. **Asynchronous Feedback Loop**: These consumers process the event concurrently and send their analyzed results back to the **User Proxy Agent** mailbox, showcasing a seamless pub-sub fan-out combined with point-to-point coordinator feedback.

## 1. Imports and Bootstrap

In [1]:
import asyncio
from ravi.kernel.runtime import LocalRuntime, AgentId, MessageContext, TopicId
from ravi.kernel.messages.content import TextBlock

print("✅ Core runtime imports ready!")

✅ Core runtime imports ready!


## 2. Define the Agent Handlers

We will define the subscriber agents and the User Proxy.

In [2]:
# A global container for us to inspect results in this notebook
user_feed_feedbacks = []

# -----------------------------------------------------
# 1. Subscriber: Sentiment Analyzer Agent
# -----------------------------------------------------
async def sentiment_handler(ctx: MessageContext, content: list[TextBlock]) -> None:
    text = content[0].text if content else ""
    print(f"[Sentiment Agent] Asynchronously analyzing: '{text}'")
    
    # Core logic: check keywords
    sentiment = "Positive 😊" if any(w in text.lower() for w in ["love", "great", "awesome", "hello"]) else "Neutral 😐"
    feedback = f"[Sentiment analysis] => {sentiment}"
    
    # Send the result back to the User Proxy! (Point-to-point)
    await ctx.runtime.send_message(
        message=feedback,
        sender=ctx.agent_id,
        recipient=AgentId(type="user_proxy", key="human")
    )

# -----------------------------------------------------
# 2. Subscriber: Moderator Agent
# -----------------------------------------------------
async def moderator_handler(ctx: MessageContext, content: list[TextBlock]) -> None:
    text = content[0].text if content else ""
    print(f"[Moderator Agent] Asynchronously checking guidelines for: '{text}'")
    
    # Core logic: safety check
    violates = "FAIL ❌ (Blocked keywords detected)" if "spam" in text.lower() else "PASS ✅"
    feedback = f"[Safety Moderation] => Status: {violates}"
    
    await ctx.runtime.send_message(
        message=feedback,
        sender=ctx.agent_id,
        recipient=AgentId(type="user_proxy", key="human")
    )

# -----------------------------------------------------
# 3. Subscriber: SEO Auto-Tagger Agent
# -----------------------------------------------------
async def seo_handler(ctx: MessageContext, content: list[TextBlock]) -> None:
    text = content[0].text if content else ""
    print(f"[SEO Agent] Asynchronously extracting keywords from: '{text}'")
    
    # Core logic: extract words longer than 5 chars as tags
    words = [w.strip("?!.,").lower() for w in text.split() if len(w) > 5]
    tags = ", ".join([f"#{w}" for w in words]) if words else "#general"
    feedback = f"[SEO Engine] => Suggested Tags: {tags}"
    
    await ctx.runtime.send_message(
        message=feedback,
        sender=ctx.agent_id,
        recipient=AgentId(type="user_proxy", key="human")
    )

# -----------------------------------------------------
# 4. User Proxy Agent (Receives the combined feedback)
# -----------------------------------------------------
async def user_proxy_handler(ctx: MessageContext, content: list[TextBlock]) -> None:
    feedback_text = content[0].text if content else ""
    print(f"[User Proxy] Received report from {ctx.sender.type}: '{feedback_text}'")
    user_feed_feedbacks.append(f"{ctx.sender.type}: {feedback_text}")

## 3. Execution Pipeline

Let's wire up the pub-sub topic registration, bootstrap the actors, publish a post from our User Proxy, and let the reactive pipeline execute!

In [3]:
async def run_social_simulation():
    user_feed_feedbacks.clear()
    
    # Initialize runtime
    runtime = LocalRuntime()
    await runtime.start()
    
    # 1. Register agents
    await runtime.register("user_proxy", user_proxy_handler)
    await runtime.register("sentiment_analyzer", sentiment_handler)
    await runtime.register("moderator", moderator_handler)
    await runtime.register("seo_tagger", seo_handler)
    
    # 2. Define our social feed event topic
    feed_topic = TopicId(type="social_events", source="global_feed")
    
    # 3. Subscribe the independent consumer agents to the feed topic
    await runtime.subscribe("sentiment_analyzer", feed_topic)
    await runtime.subscribe("moderator", feed_topic)
    await runtime.subscribe("seo_tagger", feed_topic)
    
    # --- Post 1: A Positive/Guide Post ---
    print("\n=== Action: User Proxy posts a new guide ===")
    post_content = "This new custom modular agent runtime is absolutely awesome and simple!"
    
    # Publish the post to the event topic stream
    await runtime.publish_message(
        message=post_content,
        sender=AgentId(type="user_proxy", key="human"),
        topic=feed_topic
    )
    
    # Allow async consumer tasks to react and post back reports
    await asyncio.sleep(0.2)
    
    print(f"\n--- User Proxy Aggregated Feed Results ---")
    for item in user_feed_feedbacks:
        print(f"  * {item}")
        
    await runtime.stop()

# Execute simulation
await run_social_simulation()


=== Action: User Proxy posts a new guide ===
[Sentiment Agent] Asynchronously analyzing: 'This new custom modular agent runtime is absolutely awesome and simple!'
[Moderator Agent] Asynchronously checking guidelines for: 'This new custom modular agent runtime is absolutely awesome and simple!'
[SEO Agent] Asynchronously extracting keywords from: 'This new custom modular agent runtime is absolutely awesome and simple!'
[User Proxy] Received report from sentiment_analyzer: '[Sentiment analysis] => Positive 😊'
[User Proxy] Received report from moderator: '[Safety Moderation] => Status: PASS ✅'
[User Proxy] Received report from seo_tagger: '[SEO Engine] => Suggested Tags: #custom, #modular, #runtime, #absolutely, #awesome, #simple'

--- User Proxy Aggregated Feed Results ---
  * sentiment_analyzer: [Sentiment analysis] => Positive 😊
  * moderator: [Safety Moderation] => Status: PASS ✅
  * seo_tagger: [SEO Engine] => Suggested Tags: #custom, #modular, #runtime, #absolutely, #awesome, #simp

## 4. Why this Hybrid Model Wins
* **Zero Overengineering**: The agent logic uses standard direct message loops (`send_message`), while the infrastructure/system-level events use a standard topic fan-out (`publish_message`). 
* **Clear Separation**: Reasoning and direct interaction remain deterministically ordered, whereas secondary pipelines (tagging, sentiment, safety monitoring) are decoupled. This prevents event storm loops and keeps your codebase production-grade, maintainable, and extremely clean.